[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/em_waves.ipynb)

# Electromagnetic waves full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the level of the course's worked problems) and **tier 3** stretch problems (run a formula
backwards, combine two ideas, spot the trap). Each tier is harder than the last.

This notebook also has the course's **simulations and sums**: a capacitor charging through a resistor, with the
current in its wires and the displacement current in its gap; a wave made by nothing but the two laws, stepped along a
grid and timed; the wave's two fields side by side; the whole spectrum on a powers-of-ten chart; sunlight's fields and
its push on a sail; and the sensor's three radio bands. You can move all of them yourself: the **resistor** and the
**battery**, the **material** the wave runs through, the **frequency**, the **intensity** and the sail's **area**, and
the size of the sensor's **box**.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Give the answer
in the unit the question asks for, to two decimals or more. Big and small numbers in E notation: `3e8` for 3 × 10⁸,
`8.85e-12` for 8.85 × 10⁻¹² (`3*10^8` works too). You can type the arithmetic: `3e8/90e6`, `150/3e8*1e6`,
`sqrt(2*1361/(8.85e-12*3e8))`. Numbers without thousands separators: `1012.54`, not `1,012.54`. For several answers,
separate them with commas, in the order the question lists them (the box's label shows the order). You can also put
your answer between the quotes in a cell's last line, `practice("P1.1", "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every formula and constant from the course on two pages:
[electromagnetic waves cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/em_waves_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the constants, the simulator and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, IntSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # wires and curves in chalk, labels dim, ghost lines
SKY, MINT, CORAL, VIOLET, GOLD = "#6fb6bb", "#a0c27c", "#e3896b", "#b49ad6", "#e8c56b"
# the electric field CORAL · the magnetic field MINT · the direction of travel SKY · energy VIOLET · the one mark GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="simulated · checked against the formula"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- the constants (the course's rounded values; the same in every question) ----
C = 3.00e8                 # m/s, the speed of light: three hundred million metres per second
MU0 = 4 * math.pi * 1e-7   # T m/A, mu nought: four pi times ten to the minus seven
EPS0 = 8.85e-12            # C²/(N m²), epsilon nought
C0 = 1 / math.sqrt(MU0 * EPS0)   # the speed those two constants give: 2.9986e8 m/s
g = 9.8                    # m/s²

# ---- the course's simulator: fourth-order Runge–Kutta, a fixed step at a time ----
def simulate(deriv, state, t_end, dt=1e-3, stop=None):
    """deriv(t, s) -> ds/dt as a tuple; state a tuple of numbers. Returns the list of (t, state) samples from 0 to
    t_end, or up to the first step where stop(t, s) is True."""
    t, s = 0.0, tuple(float(v) for v in state)
    out = [(t, s)]
    for _ in range(int(round(t_end / dt))):
        k1 = deriv(t, s)
        k2 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k1)))
        k3 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k2)))
        k4 = deriv(t + dt, tuple(a + dt * b for a, b in zip(s, k3)))
        s = tuple(a + dt / 6 * (b1 + 2 * b2 + 2 * b3 + b4) for a, b1, b2, b3, b4 in zip(s, k1, k2, k3, k4))
        t = t + dt
        out.append((t, s))
        if stop is not None and stop(t, s): return out
    return out

def close(a, b, rel=1e-4):
    """True when a simulated or summed value matches the formula (to a ten-thousandth by default)."""
    return abs(a - b) <= rel * max(abs(a), abs(b))

# ---- the course's formulas ----
def wavelength(f, v=C): return v / f                                                  # λ = c / f
def frequency(lam, v=C): return v / lam                                               # f = c / λ
def b_from_e(E, v=C): return E / v                                                    # B = E / c
def displacement_current(A, dEdt): return EPS0 * A * dEdt                             # I_d = ε₀ × rate of change of (E A)
def gap_field(I, r, R):                                                               # B round a charging capacitor's gap
    return MU0 * I * r / (2 * math.pi * R * R) if r <= R else MU0 * I / (2 * math.pi * r)
def speed_in(eps_r, mu_r=1.0): return C / math.sqrt(eps_r * mu_r)                     # v = 1/√(με) = c/√(εᵣ μᵣ)
def intensity(E0): return 0.5 * EPS0 * C * E0 * E0                                    # I = ½ ε₀ c E₀²
def field_from_intensity(I): return math.sqrt(2 * I / (EPS0 * C))                     # E₀ = √(2I / ε₀c)
def spread(P, d): return P / (4 * math.pi * d * d)                                    # I = P / (4π d²)
def pressure(I, mirror=False): return (2.0 if mirror else 1.0) * I / C                # radiation pressure: I/c, or 2I/c
def quarter_wave(f, v=C): return v / f / 4                                            # a quarter-wave antenna's length

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "exp": math.exp, "cos": math.cos, "sin": math.sin}

def calc(text):
    """Plain arithmetic only: numbers (3e8 too), + − × ÷, powers, brackets, sqrt, exp, cos, sin (radians) and pi."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id in ("pi", "π"): return math.pi
        if isinstance(n, ast.Name) and n.id == "e": return math.e
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNIT = (r"\s*(W/m²|W/m2|V/m|mV/m|m/s|µPa|uPa|Pa|µN|uN|mN|N|µT|uT|nT|T|GHz|MHz|kHz|Hz|m²|m2|cm|mm|nm|µm|um|km|m|mA|A|"
        r"mW|W|V|%|times)\s*$")

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    s = s.replace("√", "sqrt")
    t = re.sub(UNIT, "", s).strip()
    try: return calc(t)
    except Exception: return s

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the course's questions and answers) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Electromagnetic waves',
     "q": 'A radio station broadcasts at 90 MHz. What is its wavelength, in metres? (c = 3.00 × 10⁸ m/s)',
     "parts": ['wavelength (m)'],
     "answer": [3.3333333333333335], "tol": [0.005], "function": '/',
     "hint": 'c = f λ, so λ = c ÷ f',
     "working": 'λ = c/f = 3.00 × 10⁸ / 90 × 10⁶ = 3.33 m'},
    {"id": 'P1.2', "tier": 1, "group": 'Electromagnetic waves',
     "q": 'Orange light has a wavelength of 600 nm. What is its frequency, in units of 10¹⁴ Hz?',
     "parts": ['frequency (× 10¹⁴ Hz)'],
     "answer": [5.0], "tol": [0.005], "function": '/',
     "hint": 'f = c ÷ λ, with λ in metres (600 nm = 600e-9 m)',
     "working": 'f = c/λ = 3.00 × 10⁸ / 600 × 10⁻⁹ = 5.00 × 10¹⁴ Hz'},
    {"id": 'P1.3', "tier": 1, "group": 'Electromagnetic waves',
     "q": 'The electric field in a wave peaks at 150 V/m. What does its magnetic field peak at, in microtesla?',
     "parts": ['magnetic field (µT)'],
     "answer": [0.5], "tol": [0.005], "function": '/',
     "hint": 'E = c B, so B = E ÷ c',
     "working": 'B₀ = E₀/c = 150 / 3.00 × 10⁸ = 5.0 × 10⁻⁷ T = 0.50 µT'},
    {"id": 'P1.4', "tier": 1, "group": 'Electromagnetic waves',
     "q": 'The magnetic field in a wave peaks at 2.0 µT. What does its electric field peak at, in V/m?',
     "parts": ['electric field (V/m)'],
     "answer": [600.0], "tol": [0.005], "function": '—',
     "hint": 'E = c B',
     "working": 'E₀ = c B₀ = 3.00 × 10⁸ × 2.0 × 10⁻⁶ = 600 V/m'},
    {"id": 'P1.5', "tier": 1, "group": 'Displacement current',
     "q": 'Between two plates of area 0.02 m², the electric field grows by 5.0 × 10¹² V/m every second. What is the displacement current, in amps? (ε₀ = 8.85 × 10⁻¹² C²/(N m²))',
     "parts": ['displacement current (A)'],
     "answer": [0.885], "tol": [0.005], "function": '—',
     "hint": 'the flux is E × A; I_d = ε₀ × its rate of change',
     "working": 'I_d = ε₀ × A × (rate of change of E) = 8.85 × 10⁻¹² × 0.02 × 5.0 × 10¹² = 0.885 A'},
    {"id": 'P1.6', "tier": 1, "group": 'Displacement current',
     "q": 'A capacitor is being charged by a current of 0.30 A in its wires. What is the displacement current in the gap between its plates, in amps?',
     "parts": ['displacement current (A)'],
     "answer": [0.3], "tol": [0.005], "function": '—',
     "hint": "the gap's current matches the wire's",
     "working": 'the displacement current in the gap equals the current in the wires at every instant: 0.30 A'},
    {"id": 'P1.7', "tier": 1, "group": 'Electromagnetic waves',
     "q": 'Compute 1/√(μ₀ε₀) with μ₀ = 4π × 10⁻⁷ T m/A and ε₀ = 8.85 × 10⁻¹² C²/(N m²). Give it in units of 10⁸ m/s.',
     "parts": ['speed (× 10⁸ m/s)'],
     "answer": [2.998633804661276], "tol": [0.005], "function": 'sqrt, pi',
     "hint": 'multiply, take the square root, then one over it',
     "working": 'μ₀ε₀ = 1.112 × 10⁻¹⁷; its square root is 3.335 × 10⁻⁹; one over that is 2.9986e+08 m/s'},
    {"id": 'P1.8', "tier": 1, "group": 'Energy and momentum',
     "q": 'Sunlight of intensity 1361 W/m² is fully absorbed by a black surface. What pressure does it put on the surface, in micropascals?',
     "parts": ['pressure (µPa)'],
     "answer": [4.536666666666667], "tol": [0.005], "function": '/',
     "hint": 'absorbed light: pressure = intensity ÷ c',
     "working": 'pressure = I/c = 1361 / 3.00 × 10⁸ = 4.54 × 10⁻⁶ Pa = 4.54 µPa'},
    {"id": 'P1.9', "tier": 1, "group": 'The spectrum',
     "q": 'A sensor uses the 915 MHz band. How long is a quarter-wave antenna for it, in cm?',
     "parts": ['antenna length (cm)'],
     "answer": [8.19672131147541], "tol": [0.005], "function": '/',
     "hint": 'find the wavelength first, then divide by four',
     "working": 'λ = c/f = 3.00 × 10⁸ / 915 × 10⁶ = 0.3279 m; a quarter of it is 8.20 cm'},
    {"id": 'P1.10', "tier": 1, "group": 'The spectrum',
     "q": 'A radar sends waves 3.0 cm long. What is their frequency, in gigahertz?',
     "parts": ['frequency (GHz)'],
     "answer": [10.0], "tol": [0.005], "function": '/',
     "hint": 'f = c ÷ λ; 1 GHz is 1e9 Hz',
     "working": 'f = c/λ = 3.00 × 10⁸ / 0.030 = 1.0 × 10¹⁰ Hz = 10 GHz (microwaves)'},
    {"id": 'P2.1', "tier": 2, "group": 'Displacement current',
     "q": 'A capacitor has round plates of radius 6 cm and is being charged by 0.9 A. What is the magnetic field in the gap, 2 cm from the centre line, in microtesla?',
     "parts": ['field (µT)'],
     "answer": [1.0], "tol": [0.005], "function": '—',
     "hint": 'a loop of radius r holds the fraction r²/R² of the displacement current',
     "working": 'inside the gap only the field within your loop counts: B = μ₀ I r/(2π R²) = 2 × 10⁻⁷ × 0.9 × 0.02 / 0.0036 = 1.0 µT'},
    {"id": 'P2.2', "tier": 2, "group": 'Displacement current',
     "q": 'The same capacitor (round plates of radius 6 cm, charged by 0.9 A): what is the magnetic field 12 cm from the centre line, outside the plates, in microtesla?',
     "parts": ['field (µT)'],
     "answer": [1.5], "tol": [0.005], "function": '—',
     "hint": "outside, it is the long wire's formula",
     "working": 'outside the plates the loop holds all of it: B = μ₀ I/(2π r) = 2 × 10⁻⁷ × 0.9 / 0.12 = 1.5 µT'},
    {"id": 'P2.3', "tier": 2, "group": 'Electromagnetic waves',
     "q": 'The magnetic field in a wave peaks at 120 nT. What does its electric field peak at, in V/m?',
     "parts": ['electric field (V/m)'],
     "answer": [36.0], "tol": [0.005], "function": '—',
     "hint": 'E = c B; 1 nT is 1e-9 T',
     "working": 'E₀ = c B₀ = 3.00 × 10⁸ × 120 × 10⁻⁹ = 36 V/m'},
    {"id": 'P2.4', "tier": 2, "group": 'Electromagnetic waves',
     "q": 'In a plastic, ε is 4 times ε₀ and μ is μ₀. How fast does a wave travel in it (in units of 10⁸ m/s), and what is the wavelength of a 500 MHz wave inside it (in cm)?',
     "parts": ['speed (× 10⁸ m/s)', 'wavelength (cm)'],
     "answer": [1.5, 30.0], "tol": [0.005, 0.005], "function": 'sqrt',
     "hint": 'v = 1/√(με) = c ÷ √4; the frequency stays the same inside',
     "working": 'v = c/√4 = 1.5 × 10⁸ m/s; λ = v/f = 1.5 × 10⁸ / 500 × 10⁶ = 0.30 m = 30 cm (in air it would be 60 cm)'},
    {"id": 'P2.5', "tier": 2, "group": 'Energy and momentum',
     "q": "The electric field in a wave peaks at 50 V/m. What is the wave's intensity, in W/m²?",
     "parts": ['intensity (W/m²)'],
     "answer": [3.31875], "tol": [0.005], "function": '—',
     "hint": 'I = ½ ε₀ c E₀²',
     "working": 'I = ½ ε₀ c E₀² = 0.5 × 8.85 × 10⁻¹² × 3.00 × 10⁸ × 2500 = 3.3188 W/m²'},
    {"id": 'P2.6', "tier": 2, "group": 'Energy and momentum',
     "q": 'Sunlight above the atmosphere has an intensity of 1361 W/m². What do its electric field (V/m) and its magnetic field (µT) peak at?',
     "parts": ['electric field (V/m)', 'magnetic field (µT)'],
     "answer": [1012.5390880832314, 3.375130293610771], "tol": [0.005, 0.005], "function": 'sqrt',
     "hint": 'run I = ½ ε₀ c E₀² backwards, then B = E ÷ c',
     "working": 'E₀ = √(2I/ε₀c) = √(2 × 1361 / (8.85 × 10⁻¹² × 3.00 × 10⁸)) = 1012.54 V/m; B₀ = E₀/c = 3.3751 µT'},
    {"id": 'P2.7', "tier": 2, "group": 'Energy and momentum',
     "q": 'A small lamp sends out 5 W of light equally in all directions. What is the intensity 2 m from it, in W/m²?',
     "parts": ['intensity (W/m²)'],
     "answer": [0.0994718394324346], "tol": [0.005], "function": 'pi',
     "hint": 'the power is spread over a sphere of area 4π d²',
     "working": 'I = P/(4π d²) = 5 / (4π × 4) = 0.0995 W/m²'},
    {"id": 'P2.8', "tier": 2, "group": 'Energy and momentum',
     "q": 'Sunlight of 1361 W/m² falls square on a black panel of area 2 m², which absorbs it all. What is the force on the panel, in micronewtons?',
     "parts": ['force (µN)'],
     "answer": [9.073333333333334], "tol": [0.005], "function": '/',
     "hint": 'pressure = I ÷ c, then × the area',
     "working": 'force = (I/c) × area = 4.537 × 10⁻⁶ × 2 = 9.07 × 10⁻⁶ N = 9.07 µN'},
    {"id": 'P2.9', "tier": 2, "group": 'The spectrum',
     "q": 'A wave has a wavelength of 0.10 nm, about the width of an atom. What is its frequency, in units of 10¹⁸ Hz? (Which band is it in?)',
     "parts": ['frequency (× 10¹⁸ Hz)'],
     "answer": [3.0], "tol": [0.005], "function": '/',
     "hint": 'f = c ÷ λ; 1 nm is 1e-9 m',
     "working": 'f = c/λ = 3.00 × 10⁸ / 0.10 × 10⁻⁹ = 3.0 × 10¹⁸ Hz: X-rays'},
    {"id": 'P2.10', "tier": 2, "group": 'The spectrum',
     "q": 'A phone uses a band at 1.8 GHz. What is the wavelength (cm), and how long is a quarter-wave antenna for it (cm)?',
     "parts": ['wavelength (cm)', 'antenna length (cm)'],
     "answer": [16.666666666666664, 4.166666666666666], "tol": [0.005, 0.005], "function": '/',
     "hint": 'λ = c ÷ f, then divide by four',
     "working": 'λ = c/f = 3.00 × 10⁸ / 1.8 × 10⁹ = 0.1667 m = 16.67 cm; a quarter of it is 4.17 cm'},
    {"id": 'P3.1', "tier": 3, "group": 'Displacement current',
     "q": "At the rim of a charging capacitor's gap (round plates of radius 4 cm) the magnetic field is 0.5 µT. What current is charging it (A), and how fast is the electric field between the plates growing (in units of 10¹² V/m each second)?",
     "parts": ['current (A)', 'rate (× 10¹² V/m per s)'],
     "answer": [0.09999999999999999, 2.2479511736143407], "tol": [0.005, 0.005], "function": 'pi',
     "hint": "run the rim's field backwards to the current, then I_d = ε₀ A × rate",
     "working": "at the rim B = μ₀ I/(2π R), so I = B × 2π R/μ₀ = 0.5 × 10⁻⁶ × 0.04 / 2 × 10⁻⁷ = 0.1 A; that is ε₀ × area × the field's rate: rate = 0.1 / (8.85 × 10⁻¹² × π × 0.04²) = 2.2480e+12 V/m each second"},
    {"id": 'P3.2', "tier": 3, "group": 'Displacement current',
     "q": "A capacitor is charged through 18 Ω from a 9 V battery. One time constant (RC) after the switch closes, the wire's current has fallen to 1/e of its first value. What is the displacement current in the gap at that moment, in amps?",
     "parts": ['displacement current (A)'],
     "answer": [0.18393972058572117], "tol": [0.005], "function": 'exp',
     "hint": "the gap's current always matches the wire's",
     "working": "the first current is 9/18 = 0.5 A; after one time constant it is 0.5/e = 0.1839 A, and the gap's displacement current equals it"},
    {"id": 'P3.3', "tier": 3, "group": 'Electromagnetic waves',
     "q": 'Light of wavelength 600 nm in air enters a glass in which ε is 2.25 ε₀ and μ is μ₀. Inside the glass, what are its frequency (in units of 10¹⁴ Hz) and its wavelength (nm)?',
     "parts": ['frequency (× 10¹⁴ Hz)', 'wavelength (nm)'],
     "answer": [5.0, 400.0], "tol": [0.005, 0.005], "function": 'sqrt',
     "hint": 'the trap: it is the wavelength that changes, not the frequency',
     "working": 'the frequency is set by the source and does not change: 5.00 × 10¹⁴ Hz; the speed drops to c/1.5, so the wavelength shrinks to 600/1.5 = 400 nm'},
    {"id": 'P3.4', "tier": 3, "group": 'Energy and momentum',
     "q": 'A sensor radiates 25 mW equally in all directions. At 2 km, what is the intensity (in units of 10⁻¹⁰ W/m²) and what does the electric field peak at (mV/m)?',
     "parts": ['intensity (× 10⁻¹⁰ W/m²)', 'electric field (mV/m)'],
     "answer": [4.97359197162173, 0.6120935622400577], "tol": [0.005, 0.005], "function": 'sqrt, pi',
     "hint": 'spread the power over a sphere, then run I = ½ ε₀ c E₀² backwards',
     "working": 'I = P/(4π d²) = 0.025 / (4π × 2000²) = 4.9736e-10 W/m²; E₀ = √(2I/ε₀c) = 0.6121 mV/m'},
    {"id": 'P3.5', "tier": 3, "group": 'Energy and momentum',
     "q": 'A perfect mirror sail faces the Sun where sunlight is 1361 W/m². What area gives a push of 5 mN, in m²?',
     "parts": ['area (m²)'],
     "answer": [551.0653930933138], "tol": [0.005], "function": '/',
     "hint": 'a mirror doubles the push: force = 2 I A ÷ c',
     "working": 'a mirror feels 2I/c on each square metre: area = F c/(2I) = 0.005 × 3.00 × 10⁸ / (2 × 1361) = 551.07 m²'},
    {"id": 'P3.6', "tier": 3, "group": 'Energy and momentum',
     "q": 'The electric field of a wave is doubled. By what factor does its intensity grow? And by what factor does the intensity fall if you move three times as far from a small source?',
     "parts": ['factor for the field', 'factor for the distance'],
     "answer": [4.0, 9.0], "tol": [0.005, 0.005], "function": '—',
     "hint": 'both are squares',
     "working": 'intensity goes as E₀², so 2² = 4 times; it spreads over a sphere, so 3² = 9 times weaker'},
    {"id": 'P3.7', "tier": 3, "group": 'The spectrum',
     "q": 'For the same power and the same kind of antenna, a rule of thumb says that halving the frequency doubles the range. By that rule, how many times further does 433 MHz reach than 2.4 GHz? And how many times more power does the 433 MHz receiver catch at one fixed distance?',
     "parts": ['range factor', 'power factor'],
     "answer": [5.542725173210162, 30.72180234573762], "tol": [0.005, 0.005], "function": '—',
     "hint": 'range ∝ 1/f; power caught ∝ λ²',
     "working": 'range goes as 1/f: 2400/433 = 5.54 times; caught power goes as the wavelength squared: 5.54² = 30.72 times'},
    {"id": 'P3.8', "tier": 3, "group": 'The spectrum',
     "q": "An infrared remote control's LED shines at 940 nm. What is its frequency (in units of 10¹⁴ Hz)? How many times longer is its wave than green light's, at 550 nm?",
     "parts": ['frequency (× 10¹⁴ Hz)', 'ratio of wavelengths'],
     "answer": [3.1914893617021276, 1.709090909090909], "tol": [0.005, 0.005], "function": '/',
     "hint": 'f = c ÷ λ',
     "working": 'f = c/λ = 3.00 × 10⁸ / 940 × 10⁻⁹ = 3.1915e+14 Hz; 940/550 = 1.71 times longer'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g_, w, t) for g_, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else "✓ all right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
            print(textwrap.fill(f"Hint: {q['hint']}", 110))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g_, w, t) for g_, w, t in zip(got, answers(k), Q[k]["tol"]))

assert abs(C0 / C - 1) < 5e-4
print("Ready: c =", C, "m/s; from mu nought and epsilon nought:", round(C0), "m/s")

## Tier 1 · Warm-up

One idea, one step: a wave's length from its frequency and back again, the magnetic field that goes with an electric
field, a displacement current, the speed the two constants give, sunlight's pressure, and a quarter-wave antenna.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Each formula from the course is one short function in the first cell: `wavelength(f)`, `frequency(lam)`,
`b_from_e(E)`, `displacement_current(A, dEdt)`, `pressure(I)`, `quarter_wave(f)` and the rest. Work in hertz, metres,
volts per metre and tesla (MHz is `1e6`, GHz is `1e9`, nm is `1e-9`, µT is `1e-6`), then change the unit at the end.
Each line below computes one warm-up answer and checks it against the practice set.

In [ ]:
print(f"P1.1  λ = {wavelength(90e6):.2f} m;  P1.7  1/√(μ₀ε₀) = {1 / math.sqrt(MU0 * EPS0):.4e} m/s")
assert agrees("P1.1", [wavelength(90e6)]) and agrees("P1.2", [frequency(600e-9) / 1e14])
assert agrees("P1.3", [b_from_e(150) * 1e6]) and agrees("P1.4", [2e-6 * C])
assert agrees("P1.5", [displacement_current(0.02, 5e12)]) and agrees("P1.6", [0.30])
assert agrees("P1.7", [1 / math.sqrt(MU0 * EPS0) / 1e8]) and agrees("P1.7", [3.00])        # 3.00 is accepted too
assert agrees("P1.8", [pressure(1361) * 1e6]) and agrees("P1.9", [quarter_wave(915e6) * 100])
assert agrees("P1.10", [frequency(0.03) / 1e9])
print("Every tier 1 answer, worked out in Python, matches the practice set.")

## A charging capacitor: the wire's current and the gap's

A capacitor with round plates (radius 5 cm, 1 mm apart) is charged through a resistor from a battery. The charge on
its plates is moved a step at a time, with nothing but "the current is the battery's voltage minus the capacitor's,
over the resistance". No charge crosses the gap, but the electric field in the gap grows, and ε₀ times the rate at
which its flux grows is the **displacement current**. Read off the run, it equals the current in the wires at every
instant. Change the resistor and the battery: the two curves always lie on top of each other.

In [ ]:
PLATE_R, PLATE_D = 0.05, 1e-3
PLATE_A = math.pi * PLATE_R ** 2
CAP = EPS0 * PLATE_A / PLATE_D                      # about 70 pF

def charge_run(V, R, steps=300, taus=5.0):
    tau = R * CAP
    run = simulate(lambda t, s: ((V - s[0] / CAP) / R,), (0.0,), taus * tau, tau / steps)
    ts = np.array([t for t, _ in run]); qs = np.array([s[0] for _, s in run])
    E = qs / (EPS0 * PLATE_A)                         # the field between the plates
    i_wire = (V - qs / CAP) / R                       # the current in the wires
    i_gap = EPS0 * PLATE_A * np.gradient(E, ts)       # ε₀ × the rate of change of the flux E A
    return ts, i_wire, i_gap, tau

def charge_picture(R_ohm=18.0, V=9.0):
    ts, i_wire, i_gap, tau = charge_run(V, R_ohm)
    fig, ax = plt.subplots(figsize=(10, 4.2))
    fig.subplots_adjust(left=0.1, right=0.98, top=0.84, bottom=0.16)
    ax.plot(ts * 1e9, i_wire, color=CHALK, label="current in the wires")
    ax.plot(ts * 1e9, i_gap, color=GOLD, linestyle="--", label="displacement current in the gap")
    ax.set_xlabel("time (ns)"); ax.set_ylabel("amps"); ax.legend(frameon=False)
    worst = float(np.max(np.abs(i_gap[2:-2] / i_wire[2:-2] - 1)))
    fig.suptitle(f"starts at {V / R_ohm:.3f} A · time constant {tau * 1e9:.2f} ns · the two differ by at most {worst:.0e}",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig); plt.show()
    return i_wire[0], worst

i0, worst = charge_picture(18.0, 9.0)
assert close(i0, 0.5, 1e-9) and worst < 1e-3                 # 0.5 A at the start; the gap's current matches the wire's
assert agrees("P3.2", [0.5 * math.exp(-1)])                  # one time constant later: 0.5/e amps, in the gap too

In [ ]:
interact(charge_picture, R_ohm=FloatSlider(value=18.0, min=5.0, max=100.0, step=1.0, description="resistor (Ω)"),
         V=FloatSlider(value=9.0, min=1.0, max=24.0, step=0.5, description="battery (V)"));

## Tier 2 · Core

The level of the course's worked problems: the magnetic field in and round a capacitor's gap, a wave in a material,
intensity from the field and back, light spreading from a small source, the push of sunlight, and bands of the
spectrum.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

Two-part answers are just two lines. Inside the gap, only the growing field within your loop counts, so
`gap_field(I, r, R)` rises in step with r up to the rim and then falls like a wire's field.

In [ ]:
E_sun = field_from_intensity(1361)
print(f"P2.6  E₀ = {E_sun:.2f} V/m, B₀ = {b_from_e(E_sun) * 1e6:.2f} µT;  P2.5  I = {intensity(50):.2f} W/m²")
assert agrees("P2.1", [gap_field(0.9, 0.02, 0.06) * 1e6]) and agrees("P2.2", [gap_field(0.9, 0.12, 0.06) * 1e6])
assert agrees("P2.3", [120e-9 * C]) and agrees("P2.4", [speed_in(4) / 1e8, wavelength(500e6, speed_in(4)) * 100])
assert agrees("P2.5", [intensity(50)]) and agrees("P2.6", [E_sun, b_from_e(E_sun) * 1e6])
assert agrees("P2.7", [spread(5, 2)]) and agrees("P2.8", [pressure(1361) * 2 * 1e6])
assert agrees("P2.9", [frequency(0.1e-9) / 1e18]) and agrees("P2.10", [wavelength(1.8e9) * 100, quarter_wave(1.8e9) * 100])
print("Every tier 2 answer, worked out in Python, matches the practice set.")

## A wave from two laws

This cell knows no formula for a wave and no speed of light. It holds the electric field E at a row of points and the
magnetic field B half-way between them, and steps both forward with the two laws alone: Faraday's law (B changes where
E varies along the row) and the Ampère-Maxwell law (E changes where B varies along the row, with μ₀ε₀ in it). Start
with one bump of E and no B at all. The bump splits into two half-height bumps that travel apart. Time one of them:
its speed is 1/√(μ₀ε₀). Its E divided by its B is the same number, c. Change the material (ε in units of ε₀) and the
wave slows to c/√ε.

In [ ]:
def grid_run(steps, eps_r=1.0, n=3000, dx=0.01, x0=15.0, sigma=0.4):
    dt = 0.5 * dx / C0
    xe = np.arange(n) * dx; xb = xe[:-1] + dx / 2
    E = np.exp(-(xe - x0) ** 2 / (2 * sigma ** 2)); B = np.zeros(n - 1)
    for _ in range(steps):
        B += dt / dx * (E[1:] - E[:-1])                                   # Faraday's law
        E[1:-1] += dt / (MU0 * EPS0 * eps_r * dx) * (B[1:] - B[:-1])      # the Ampère-Maxwell law, no charge moving
    return xe, E, xb, B, steps * dt

def peak(x, y):
    k = int(np.argmax(y)); a, b, c_ = y[k - 1], y[k], y[k + 1]
    return x[k] + 0.5 * (a - c_) / (a - 2 * b + c_) * (x[1] - x[0])

def wave_picture(eps_r=1.0, steps=1200):
    xe, E, xb, B, t = grid_run(int(steps), eps_r)
    right_half = xe > 15.0
    xp = peak(xe[right_half], E[right_half])
    v = (xp - 15.0) / t
    kb = int(np.argmin(np.abs(xb - xp)))
    fig, axes = plt.subplots(2, 1, figsize=(10, 5.2), sharex=True)
    fig.subplots_adjust(left=0.09, right=0.98, top=0.86, bottom=0.12, hspace=0.18)
    axes[0].plot(xe, E, color=CORAL); axes[0].set_ylabel("E (the start bump = 1)")
    axes[1].plot(xb, B * C0, color=MINT); axes[1].set_ylabel("B × c"); axes[1].set_xlabel("metres along the grid")
    axes[0].axvline(15.0, color=GHOST); axes[1].axvline(15.0, color=GHOST)
    fig.suptitle(f"after {t * 1e9:.1f} ns the right-hand bump has moved {xp - 15:.2f} m: {v:.4e} m/s "
                 f"(c/√ε = {C0 / math.sqrt(eps_r):.4e})", x=0.03, ha="left", color=GOLD)
    footnote(fig, "E up: B is negative (towards you) on the bump going right, positive on the one going left")
    plt.show()
    return v, float(E[right_half].max()), float(E[right_half].max() / abs(B[kb]))

v_sim, height, ratio = wave_picture(1.0, 1200)
assert close(v_sim, C0, 1e-3) and close(v_sim, 299792458, 1e-3)      # the two laws alone give the speed of light
assert close(height, 0.5, 5e-3) and close(ratio, C0, 5e-3)           # two half bumps; E = c B
v_glass, _, _ = wave_picture(2.25, 1200)
assert close(v_glass, C0 / 1.5, 3e-3)                                 # ε = 2.25 ε₀: two thirds of c

In [ ]:
interact(wave_picture, eps_r=FloatSlider(value=1.0, min=1.0, max=9.0, step=0.25, description="ε (in ε₀)"),
         steps=IntSlider(value=1200, min=200, max=2400, step=200, description="steps"));

## The wave's two fields

A sine wave travelling to the right: the electric field up and down, the magnetic field towards you and away, in step,
with E = c B at every point. Choose the frequency and the peak electric field: the wavelength follows from c = f λ, and
the peak magnetic field from B = E/c. (The magnetic curve is drawn times c, so the two can share one scale.)

In [ ]:
def fields_picture(f_MHz=100.0, E0=300.0):
    f = f_MHz * 1e6
    lam = wavelength(f)
    x = np.linspace(0, 3 * lam, 600)
    E = E0 * np.sin(2 * np.pi * x / lam)
    B = -b_from_e(E)                                    # towards you (negative y) where E is up: E × B points along the travel
    fig, ax = plt.subplots(figsize=(10, 4.0))
    fig.subplots_adjust(left=0.1, right=0.98, top=0.84, bottom=0.17)
    ax.plot(x, E, color=CORAL, label="E (V/m), up and down")
    ax.plot(x, B * C, color=MINT, linestyle="--", label="B × c, towards you and away")
    ax.set_xlabel("metres along the travel"); ax.legend(frameon=False, loc="upper right")
    fig.suptitle(f"{f_MHz:g} MHz: λ = {lam:.4g} m · E₀ = {E0:g} V/m goes with B₀ = {b_from_e(E0) * 1e6:.4g} µT",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig, "our own example · computed"); plt.show()
    return lam, b_from_e(E0)

lam_fm, b_fm = fields_picture(100.0, 300.0)
assert close(lam_fm, 3.0, 1e-12) and close(b_fm, 1e-6, 1e-12)        # 100 MHz: 3 m; 300 V/m goes with 1 µT
assert agrees("P1.3", [b_from_e(150) * 1e6])

In [ ]:
interact(fields_picture, f_MHz=FloatSlider(value=100.0, min=10.0, max=3000.0, step=10.0, description="f (MHz)"),
         E0=FloatSlider(value=300.0, min=10.0, max=1500.0, step=10.0, description="E₀ (V/m)"));

## Tier 3 · Stretch

Run a formula backwards, combine two ideas, or spot the trap: the field's growth from the magnetic field at the rim,
the gap's current one time constant on, light entering glass, a sensor's field two kilometres away, a sail's area, two
squares, the range rule, and a remote control's frequency.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

Running a formula backwards is one line of algebra, then one line of Python. The trap in P3.3: when a wave enters a
material its frequency stays the same, and it is the wavelength that shrinks.

In [ ]:
I_rim = 0.5e-6 * 2 * math.pi * 0.04 / MU0
assert agrees("P3.1", [I_rim, I_rim / (EPS0 * math.pi * 0.04 ** 2) / 1e12]) and agrees("P3.2", [9 / 18 * math.exp(-1)])
assert agrees("P3.3", [frequency(600e-9) / 1e14, 600 / math.sqrt(2.25)])
I_far = spread(0.025, 2000)
print(f"P3.4  I = {I_far:.3e} W/m², E₀ = {field_from_intensity(I_far) * 1e3:.2f} mV/m")
assert agrees("P3.4", [I_far * 1e10, field_from_intensity(I_far) * 1e3])
assert agrees("P3.5", [5e-3 * C / (2 * 1361)]) and agrees("P3.6", [2 ** 2, 3 ** 2])
assert agrees("P3.7", [2.4e9 / 433e6, (2.4e9 / 433e6) ** 2]) and agrees("P3.8", [frequency(940e-9) / 1e14, 940 / 550])
print("Every tier 3 answer, worked out in Python, matches the practice set.")

## The spectrum on a powers-of-ten chart

Every kind of electromagnetic wave on one chart, from 10⁴ Hz to 10²² Hz. Each step along it multiplies the frequency
by ten. Slide the marker: the chart gives the wavelength (c = f λ) and the band. The band edges are conventions, and
neighbours overlap. Visible light is the thin gold sliver.

In [ ]:
BAND_NAMES = ["radio waves", "microwaves", "infrared", "visible light", "ultraviolet", "X-rays", "gamma rays"]
BAND_EDGES_M = [1.0, 1e-3, 700e-9, 400e-9, 10e-9, 0.01e-9]            # the wavelength between each pair of neighbours
BAND_EDGES_HZ = [frequency(l_) for l_ in BAND_EDGES_M]

def band(f):
    return BAND_NAMES[sum(1 for e_ in BAND_EDGES_HZ if f >= e_)]

def nice_length(lam):
    for unit, size in (("km", 1e3), ("m", 1.0), ("cm", 1e-2), ("mm", 1e-3), ("µm", 1e-6), ("nm", 1e-9), ("pm", 1e-12)):
        if lam >= size: return f"{lam / size:.3g} {unit}"
    return f"{lam:.3g} m"

def spectrum_picture(log_f=14.74):
    f = 10 ** log_f
    fig, ax = plt.subplots(figsize=(10.5, 2.8))
    fig.subplots_adjust(left=0.03, right=0.97, top=0.72, bottom=0.3)
    edges = [4] + [math.log10(e_) for e_ in BAND_EDGES_HZ] + [22]
    for k, name in enumerate(BAND_NAMES):
        col = GOLD if name == "visible light" else (GHOST if k % 2 else "#2c3c38")
        ax.axvspan(edges[k], edges[k + 1], color=col)
        if name != "visible light":
            ax.text((edges[k] + edges[k + 1]) / 2, 0.5, name, ha="center", va="center", color=CHALK, fontsize=11)
    ax.axvline(log_f, color=CORAL)
    ax.set_xlim(4, 22); ax.set_ylim(0, 1); ax.set_yticks([]); ax.grid(False)
    ax.set_xticks(range(4, 23, 2)); ax.set_xticklabels([f"10^{k}" for k in range(4, 23, 2)]); ax.set_xlabel("frequency (Hz)")
    fig.suptitle(f"{f:.3g} Hz · wavelength {nice_length(wavelength(f))} · {band(f)}", x=0.03, ha="left", color=GOLD)
    footnote(fig, "band edges are conventions · wavelengths from c = f λ"); plt.show()
    return wavelength(f), band(f)

assert spectrum_picture(math.log10(frequency(550e-9)))[1] == "visible light"
assert [band(f_) for f_ in (1e6, 100e6, 2.45e9, frequency(940e-9), frequency(254e-9), frequency(0.1e-9), 1e21)] == [
    "radio waves", "radio waves", "microwaves", "infrared", "ultraviolet", "X-rays", "gamma rays"]
assert 700 / 400 < 2 and math.log10(700 / 400) / 18 < 1 / 50          # visible light: not even one doubling of frequency
assert agrees("P2.9", [frequency(0.1e-9) / 1e18]) and band(frequency(10e-6)) == "infrared"

In [ ]:
interact(spectrum_picture, log_f=FloatSlider(value=14.74, min=4.0, max=22.0, step=0.02, description="log₁₀ f"));

## Sunlight's fields, and its push on a sail

Intensity is the power a wave brings to each square metre: ½ ε₀ c E₀². Run that backwards and sunlight's 1361 W/m²
(above the atmosphere) turns out to carry an electric field of about a thousand volts per metre. A wave also carries
momentum, its energy divided by c, so it pushes: I/c on a surface that absorbs it, twice that on a mirror. Slide the
intensity and a mirror sail's area.

In [ ]:
def sun_picture(I=1361.0, area=196.0):
    E0 = field_from_intensity(I); B0 = b_from_e(E0)
    F = pressure(I, mirror=True) * area
    areas = np.linspace(1, 1000, 200)
    fig, ax = plt.subplots(figsize=(10, 3.8))
    fig.subplots_adjust(left=0.1, right=0.98, top=0.82, bottom=0.18)
    ax.plot(areas, pressure(I, mirror=True) * areas * 1e3, color=VIOLET, label="a perfect mirror")
    ax.plot(areas, pressure(I) * areas * 1e3, color=DIM, label="a black sail")
    ax.plot([area], [F * 1e3], "o", color=GOLD, ms=9)
    ax.set_xlabel("sail area (m²)"); ax.set_ylabel("push (mN)"); ax.legend(frameon=False)
    fig.suptitle(f"{I:g} W/m²: E₀ = {E0:.0f} V/m, B₀ = {B0 * 1e6:.2f} µT · pressure {pressure(I) * 1e6:.2f} µPa · "
                 f"mirror sail of {area:g} m²: {F * 1e3:.2f} mN", x=0.03, ha="left", color=GOLD)
    footnote(fig, "our own example · computed"); plt.show()
    return E0, B0, F

E_s, B_s, F_s = sun_picture(1361.0, 196.0)
assert close(E_s, 1012.54, 1e-5) and close(B_s * 1e6, 3.375, 1e-3) and close(F_s * 1e3, 1.778, 1e-3)
assert close(intensity(E_s), 1361.0, 1e-12) and F_s > 1.12e-3      # the real 14 m sail (IKAROS) measured 1.12 mN
assert agrees("P1.8", [pressure(1361) * 1e6]) and agrees("P3.5", [5e-3 * C / (2 * 1361)])

In [ ]:
interact(sun_picture, I=FloatSlider(value=1361.0, min=100.0, max=3000.0, step=1.0, description="I (W/m²)"),
         area=FloatSlider(value=196.0, min=1.0, max=1000.0, step=1.0, description="area (m²)"));

## The sensor's three bands

The course's opening question. A small sensor can use 433 MHz, 868 MHz or 2.4 GHz. Each band's wavelength comes from
c = f λ, and a simple antenna is a quarter of that. For the same power and the same kind of antenna, a rule of thumb
from antenna engineering says that halving the frequency doubles the range in open country. So the lowest band reaches
furthest, but needs the longest antenna. Slide the size of the sensor's box: the sensor takes the lowest band whose
antenna fits.

In [ ]:
SENSOR_BANDS = {"433 MHz": 433e6, "868 MHz": 868e6, "2.4 GHz": 2.4e9}

def sensor_picture(box_cm=10.0):
    names = list(SENSOR_BANDS)
    lam = [wavelength(SENSOR_BANDS[k]) * 100 for k in names]
    quarter = [quarter_wave(SENSOR_BANDS[k]) * 100 for k in names]
    reach = [SENSOR_BANDS["2.4 GHz"] / SENSOR_BANDS[k] for k in names]
    fits = [k for k, q_ in zip(names, quarter) if q_ <= box_cm]
    choice = min(fits, key=lambda k: SENSOR_BANDS[k]) if fits else None
    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.6))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.78, bottom=0.14, wspace=0.32)
    for ax, vals, title in zip(axes, (lam, quarter, reach), ("wavelength (cm)", "quarter-wave antenna (cm)", "range, against 2.4 GHz")):
        ax.bar(names, vals, color=[GOLD if k == choice else DIM for k in names]); ax.set_title(title, fontsize=12)
    axes[1].axhline(box_cm, color=CORAL, linestyle="--")
    fig.suptitle((f"a {box_cm:g} cm box: {choice}, the lowest band whose antenna fits" if choice else
                  f"a {box_cm:g} cm box: no quarter-wave antenna fits"), x=0.03, ha="left", color=GOLD)
    footnote(fig, "our own example · computed"); plt.show()
    return lam, quarter, reach, choice

lam3, quarter3, reach3, choice = sensor_picture(10.0)
assert [round(v, 1) for v in lam3] == [69.3, 34.6, 12.5] and [round(v, 1) for v in quarter3] == [17.3, 8.6, 3.1]
assert [round(v, 1) for v in reach3] == [5.5, 2.8, 1.0] and choice == "868 MHz"
assert agrees("P3.7", [reach3[0], reach3[0] ** 2]) and agrees("P1.9", [quarter_wave(915e6) * 100])

In [ ]:
interact(sensor_picture, box_cm=FloatSlider(value=10.0, min=2.0, max=25.0, step=0.5, description="box (cm)"));

## Sources
Every exercise and set-up here is our own. The constants are the course's rounded values (c = 3.00 × 10⁸ m/s, exactly
299,792,458; μ₀ = 4π × 10⁻⁷ T m/A; ε₀ = 8.85 × 10⁻¹² C²/(N m²); CODATA 2022, physics.nist.gov/cuu/Constants). Sunlight
above the atmosphere, 1361 W/m², is the nominal value of IAU 2015 Resolution B3. The 14 m sail's measured 1.12 mN is
from JAXA's press release on IKAROS of 9 July 2010. The three bands are licence-free bands of the ITU Radio Regulations
(433 MHz in ITU Region 1; 863 to 870 MHz in Europe; 2.4 GHz worldwide), and the quarter-wave lengths and the rule
"halve the frequency, double the range" (for the same power, sensitivity and antenna gain, in line of sight) are from
Texas Instruments' application note AN058, "Antenna Selection Guide". The band edges of the spectrum are common
conventions; neighbouring bands overlap. The simulator is the course's: fourth-order Runge–Kutta with a fixed step;
the wave is stepped on a grid by the two laws alone.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib, sys
_src = pathlib.Path("../../../episodes/ph18_em_waves/practice.py")
if _src.exists():
    sys.path.insert(0, str(_src.resolve().parents[2]))
    _spec = importlib.util.spec_from_file_location("ph18_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"], _q["tier"], _q["parts"], _q["hint"]) == (
            _e["q"], _e["working"], _e["function"], _e["tier"], _e["parts"], _e["hint"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 * max(1, abs(b)) for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert len(_P.EX) == len(EX)
    assert (C, MU0, EPS0, g) == (_P.C, _P.MU0, _P.EPS0, _P.g)
    assert BAND_NAMES == _P.BAND_NAMES and BAND_EDGES_M == _P.BAND_EDGES_M and SENSOR_BANDS == _P.SENSOR_BANDS
    assert close(_P.SIM["pulse_speed"], v_sim, 1e-3) and close(_P.SUN["E0"], E_s, 1e-12) and _P.HOOK_CHOICE == choice
    print("matches the course's practice set")